In [1]:
%%writefile config.py

# File path for the dataset inside the Kaggle environment
DATA_DIR = '/kaggle/input/bone-fracture-dataset/Bone fracture dataset/Bone fracture dataset/Dataset'

# Settings for the model and training process
NUM_EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.00001

# The filename for the final saved model
MODEL_SAVE_PATH = 'bone_fracture_model_final.pth'


Writing config.py


In [2]:
%%writefile utils.py

import torch
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def get_dataloaders(data_dir, batch_size):
    """
    Loads and splits the data into training, validation, and test sets.
    """
    transforms_all = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    full_dataset = ImageFolder(data_dir, transform=transforms_all)
    
    train_size = int(0.7 * len(full_dataset))
    val_size = int(0.15 * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size
    
    torch.manual_seed(42)
    train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print("Data successfully split:")
    print(f" - Training images:   {len(train_dataset)}")
    print(f" - Validation images: {len(val_dataset)}")
    print(f" - Test images:       {len(test_dataset)}")
    
    return train_loader, val_loader, test_loader, full_dataset.classes

def plot_performance_curves(history):
    """
    Plots the performance curves (loss and accuracy) vs. epochs and saves the figure.
    """
    num_epochs = len(history['train_loss'])
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.plot(range(1, num_epochs + 1), history['train_loss'], 'o-', label='Training Loss')
    plt.plot(range(1, num_epochs + 1), history['val_loss'], 'o-', label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Loss vs. Epochs')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(range(1, num_epochs + 1), history['train_acc'], 'o-', label='Training Accuracy')
    plt.plot(range(1, num_epochs + 1), history['val_acc'], 'o-', label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy vs. Epochs')
    plt.legend()
    
    plt.tight_layout()
    # NEW: Save the figure to a file
    plt.savefig('performance_curves.png')
    plt.show()
    plt.close() # Close the figure to free up memory

def plot_confusion_matrix(model, loader, dataset_name, class_names, device):
    """
    Calculates and plots the confusion matrix for a given dataset and saves the figure.
    """
    y_pred, y_true = [], []
    model.eval()
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            y_pred.extend(predicted.cpu().numpy())
            y_true.extend(labels.cpu().numpy())
            
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix for {dataset_name} Set')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # NEW: Save the figure to a file (with a unique name)
    filename = f"confusion_matrix_{dataset_name.replace(' ', '_')}.png"
    plt.savefig(filename)
    plt.show()
    plt.close() # Close the figure to free up memory


Writing utils.py


In [3]:
%%writefile model.py

import torch
import torch.nn as nn
import torchvision.models as models

def create_model(device):
    """
    Creates and prepares a ResNet50 model for fine-tuning.
    """
    model = models.resnet50(weights='IMAGENET1K_V1')
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2) # 2 classes: fracture, normal
    model.to(device)
    
    # Unfreeze all layers for full fine-tuning
    for param in model.parameters():
        param.requires_grad = True
        
    return model


Writing model.py


In [4]:
%%writefile train.py

import torch
import torch.nn as nn
import torch.optim as optim
import time
import config
from utils import get_dataloaders, plot_performance_curves, plot_confusion_matrix
from model import create_model

def main():
    # Set up the device (GPU or CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load the data using the function from utils.py
    train_loader, val_loader, test_loader, class_names = get_dataloaders(config.DATA_DIR, config.BATCH_SIZE)

    # Create the model using the function from model.py
    model = create_model(device)

    # Define the optimizer and loss function using settings from config.py
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    # --- Start Training ---
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    print("\nStarting model training...")

    for epoch in range(config.NUM_EPOCHS):
        start_time = time.time()
        
        # Training phase
        model.train()
        running_train_loss, correct_train, total_train = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
        
        epoch_train_loss = running_train_loss / len(train_loader.dataset.indices)
        epoch_train_acc = 100 * correct_train / total_train
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # Validation phase
        model.eval()
        running_val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
        
        epoch_val_loss = running_val_loss / len(val_loader.dataset.indices)
        epoch_val_acc = 100 * correct_val / total_val
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{config.NUM_EPOCHS} [{epoch_time:.2f}s] | "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%")

    print("\n--- Training Complete ---")

    # Save the trained model
    torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
    print(f"Model saved to {config.MODEL_SAVE_PATH}")

    # --- Display Analysis ---
    plot_performance_curves(history)
    
    print("\n--- Confusion Matrices ---")
    plot_confusion_matrix(model, train_loader, "Training", class_names, device)
    plot_confusion_matrix(model, val_loader, "Validation", class_names, device)
    plot_confusion_matrix(model, test_loader, "Testing", class_names, device)

if __name__ == '__main__':
    main()


Writing train.py


In [5]:
!python train.py


Using device: cuda
Data successfully split:
 - Training images:   1488
 - Validation images: 319
 - Test images:       320
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 183MB/s]

Starting model training...
Epoch 1/10 [43.34s] | Train Loss: 0.3885, Train Acc: 94.15% | Val Loss: 0.2956, Val Acc: 96.24%
Epoch 2/10 [32.15s] | Train Loss: 0.1543, Train Acc: 98.45% | Val Loss: 0.1207, Val Acc: 98.43%
Epoch 3/10 [33.22s] | Train Loss: 0.0744, Train Acc: 99.60% | Val Loss: 0.0657, Val Acc: 99.06%
Epoch 4/10 [32.96s] | Train Loss: 0.0378, Train Acc: 99.93% | Val Loss: 0.0322, Val Acc: 100.00%
Epoch 5/10 [32.96s] | Train Loss: 0.0248, Train Acc: 99.87% | Val Loss: 0.0218, Val Acc: 100.00%
Epoch 6/10 [32.92s] | Train Loss: 0.0171, Train Acc: 100.00% | Val Loss: 0.0145, Val Acc: 100.00%
Epoch 7/10 [32.83s] | Train Loss: 0.0132, Train Acc: 99